# Prepare training data for Unsloth
Merges `train_raw.jsonl` and `data_starter.xlsx`, then writes `train_all.jsonl` (for checking) and `train_chat.jsonl` (for Unsloth). Run the cells from top to bottom.

In [1]:
import json
import collections
from openpyxl import load_workbook

DATA = "../data/"   # change this if your files are somewhere else
system = open(DATA + "system_prompt.txt", encoding="utf-8").read().strip()
print(system)

You triage customer messages for an online shop in Malaysia. Read the message and reply with ONLY a JSON object with exactly these keys: category, urgency, needs_human. category is one of: delivery, refund_return, product_fault, payment_billing, account_access, general_enquiry. urgency is one of: low, medium, high. needs_human is true or false.


## 1. Rows from `train_raw.jsonl`

In [2]:
rows = [json.loads(l) for l in open(DATA + "train_raw.jsonl", encoding="utf-8") if l.strip()]
print(len(rows), "rows from train_raw.jsonl")

157 rows from train_raw.jsonl


## 2. Rows from the Excel sheet

In [3]:
ws = load_workbook(DATA + "data_starter.xlsx", data_only=True).worksheets[0]
it = ws.iter_rows(values_only=True)
header = [str(h).strip().lower() for h in next(it)]
n_before = len(rows)
for r in it:
    d = dict(zip(header, r))
    if not d.get("message"):
        continue
    nh = d["needs_human"]
    nh = nh if isinstance(nh, bool) else str(nh).strip().lower() in ("true", "1", "yes")
    rows.append({"id": str(d["id"]), "message": str(d["message"]).strip(),
                 "category": d["category"], "urgency": d["urgency"], "needs_human": nh})
print(len(rows) - n_before, "rows from data_starter.xlsx |", len(rows), "total")

50 rows from data_starter.xlsx | 207 total


## 3. Check the data

In [4]:
CATEGORIES = {"delivery", "refund_return", "product_fault", "payment_billing", "account_access", "general_enquiry"}
URGENCIES = {"low", "medium", "high"}

bad = [r["id"] for r in rows
       if r["category"] not in CATEGORIES or r["urgency"] not in URGENCIES
       or not isinstance(r["needs_human"], bool)]
msgs = [r["message"].strip().lower() for r in rows]
dups = [m for m, c in collections.Counter(msgs).items() if c > 1]

try:
    test = {json.loads(l)["message"].strip().lower()
            for l in open(DATA + "test_raw.jsonl", encoding="utf-8") if l.strip()}
    leaked = sum(m in test for m in msgs)
except FileNotFoundError:
    leaked = "test_raw.jsonl not found, skipped"

print("bad labels:", bad)
print("duplicate messages:", len(dups))
print("overlap with test set:", leaked)
print(collections.Counter(r["category"] for r in rows))
print(collections.Counter(r["urgency"] for r in rows))
print(collections.Counter(r["needs_human"] for r in rows))

bad labels: []
duplicate messages: 0
overlap with test set: 0
Counter({'delivery': 36, 'refund_return': 35, 'payment_billing': 35, 'account_access': 35, 'general_enquiry': 34, 'product_fault': 32})
Counter({'medium': 86, 'low': 78, 'high': 43})
Counter({False: 147, True: 60})


## 4. Write the merged file (for checking)

In [5]:
with open(DATA + "train_all.jsonl", "w", encoding="utf-8") as f:
    for r in rows:
        f.write(json.dumps(r, ensure_ascii=False) + "\n")
print("wrote train_all.jsonl")

wrote train_all.jsonl


## 5. Write the chat-format file (for Unsloth)

In [6]:
with open(DATA + "train_chat.jsonl", "w", encoding="utf-8") as f:
    for r in rows:
        answer = json.dumps({"category": r["category"], "urgency": r["urgency"],
                             "needs_human": r["needs_human"]})
        f.write(json.dumps({"messages": [
            {"role": "system", "content": system},
            {"role": "user", "content": r["message"]},
            {"role": "assistant", "content": answer}]}, ensure_ascii=False) + "\n")

print(f"Done: {len(rows)} rows")

Done: 207 rows


## 6. Preview one training example

In [7]:
first = json.loads(open(DATA + "train_chat.jsonl", encoding="utf-8").readline())
for m in first["messages"]:
    print(m["role"].upper(), ":", m["content"], "\n")

SYSTEM : You triage customer messages for an online shop in Malaysia. Read the message and reply with ONLY a JSON object with exactly these keys: category, urgency, needs_human. category is one of: delivery, refund_return, product_fault, payment_billing, account_access, general_enquiry. urgency is one of: low, medium, high. needs_human is true or false. 

USER : Can I change the delivery address for order OS-20431? I have not received the shipping email yet. 

ASSISTANT : {"category": "delivery", "urgency": "low", "needs_human": false} 

